This is an interactive tutorial that will demonstrate how to run jobs with SLURM by directly running sample scripts within code blocks. Refer to the README file to follow a more rigorous tutorial for running jobs from the command line and for batch submission of jobs. This tutorial uses *Jupyter notebook* which is an awesome software for creating a script with *markdown* and *code* blocks that allow you to write visually rich comments and run code from multiple languages.

Job submissions are intended to take a process and run it remotely using CARC resources. To that end, we will first establish the process we want to run by creating the following python script. For this tutorial we will use a simple script with a toy situation that takes randomly generated train times to determine if members of the Levine lab are going to catch the train to work or not.

In [ ]:
example_python="""
#This is a small python script with a modifiable input
import pandas as pd
import numpy as np
import random

def catch_the_train(person,commute):
    #Establish the train schedule numerically and as strings
    num_train_schedule=np.array([805,815,825,835,845,855])
    train_schedule=np.array(["8:05","8:15","8:25","8:35","8:45","8:55"])

    #Establist a random time to depart for the train, and a random delay in the train arrival
    depart_time=random.randint(800,860)
    train_delay=random.randint(-2,2)
    num_train_schedule=num_train_schedule+train_delay

    #Check for special cases
    if (person == "Naomi"):
        print("Naomi doesn't take the train!\n")
        return
    if (depart_time>=855):
        print(f"{person} left too late and missed all the trains!\n")
        return

    #Compute arrival time and see when the next train is
    arrival_time=depart_time+commute
    print(f"{person} is departing at {depart_time}")
    coming_trains=[i for i in range(len(num_train_schedule)) if num_train_schedule[i] >= depart_time]
    next_train=coming_trains[0]
    next_train_time=num_train_schedule[next_train]
    next_train_str=train_schedule[next_train]

    #Check to see if the person makes the train
    if arrival_time<=next_train_time:
        print(f"{person} left at {depart_time} and made it to the {next_train_str} train just in time!\n")
    else:
        print(f"{person} left at {depart_time} and did not make the {next_train_str} train and will have to wait for the next one!\n")
    return

#Import our dataset and pass it through our function to see if we can catch 
#the train
print("Running our first job!\n")

#Read in the dataframe to run
Data=pd.read_csv("conditions.txt",sep="\t")

#The person and commute time will be directly inserted by the job generation script
for i in range(len(Data["Person"])):
    curr_person=Data["Person"][i]
    curr_commute=Data["Commute"][i]
    catch_the_train(curr_person,curr_commute)

"""

with open ('example-code.py','w') as fp:
    fp.write(example_python)


To run a job on SLURM, you need to create a job submission script which uses the special **SBATCH** expression to specify to the cluster the compute parameters you want to use while running your job. You have to specify the first two **SBATCH** lines, the **-p** and **-A** commands which let CARC know which computer cores to use and under which account you are authorized to run your jobs. Once you've defined all the parameters, you can fill out the rest of the script as you wish using bash syntax. For us, we just need to load the python module and call the example code we wrote previously.

In [ ]:
submit_script="""
#!/bin/bash
#SBATCH -p qcb
#SBATCH -A nmherrer_110
#SBATCH -t 1-00:00:00
#SBATCH --mem=16gb
#SBATCH --cpus-per-task=8
#SBATCH --out=python_out
#SBATCH --error=python_err

module load python
python example-code.py
"""

with open('submit-test_script.slurm','w') as fp:
    fp.write(submit_script)

With our submit script in hand we could now call our script from our command line interface (CLI) using the *sbatch* command. This will tell CARC to take our script, find a compute node with the right resources, and run it for us. NOTE: The exclamation point (!) here is a Jupyter notebook syntax for running a single line in bash.

In [ ]:
!sbatch submit-test_script.slurm

We can always check to see if our job is running, how many jobs we have running, etc. using pre-built CARC commands, in particular *squeue*. You can use *squeue* with the **--me** flag to look at all the jobs run from your account. You can use the **--out** and **--error** **SBATCH** flags to output any command line results or errors to named files that you can use to investigate any potential issues.

In [ ]:
!squeue --me

In addition to directly submitting jobs as we have done above, we can also perform batch submissions. Batch submissions are ideal for instances where you want to run the same process a large number of times. The key difference between direct and batch submissions is that you want to convert your process and submission scripts into *template* files. These files can no longer be run on their own, but can be modified to run using conditions specified by the user. This allows us to create *multiple* copies of our file each of which can run our process over a prescribed set of conditions. These batched processes can run in parallel, thus offering substantial speed-ups for running your scripts. Take a look at the batch python script we define below.

In [ ]:
batch_python="""
#This is a small python script with a modifiable input
import pandas as pd
import numpy as np
import random

def catch_the_train(person,commute):
    #Establish the train schedule numerically and as strings
    num_train_schedule=np.array([805,815,825,835,845,855])
    train_schedule=np.array(["8:05","8:15","8:25","8:35","8:45","8:55"])

    #Establist a random time to depart for the train, and a random delay in the train arrival
    depart_time=random.randint(800,860)
    train_delay=random.randint(-2,2)
    num_train_schedule=num_train_schedule+train_delay

    #Check for special cases
    if (person == "Naomi"):
        print("Naomi doesn't take the train!\n")
        return
    if (depart_time>=855):
        print(f"{person} left too late and missed all the trains!\n")
        return

    #Compute arrival time and see when the next train is
    arrival_time=depart_time+commute
    print(f"{person} is departing at {depart_time}")
    coming_trains=[i for i in range(len(num_train_schedule)) if num_train_schedule[i] >= depart_time]
    next_train=coming_trains[0]
    next_train_time=num_train_schedule[next_train]
    next_train_str=train_schedule[next_train]

    #Check to see if the person makes the train
    if arrival_time<=next_train_time:
        print(f"{person} left at {depart_time} and made it to the {next_train_str} train just in time!")
    else:
        print(f"{person} left at {depart_time} and did not make the {next_train_str} train and will have to wait for the next one!")
    return

#Import our dataset and pass it through our function to see if we can catch 
#the train
print("Running our first job!\n")

#The person and commute time will be directly inserted by the job generation script
curr_person="sub_person"
curr_commute=sub_commute
catch_the_train(curr_person,curr_commute)

"""

with open('batch-code.py','w') as fp:
    fp.write(batch_python)


You'll notice some instances where we have replaced certain information with dummy variables like *sub_person* and *sub_commute*. This allows us to substitute these variables at will for values from a conditions file. We will also create a template version of our *submission* script.

In [ ]:
batch_submit="""
#!/bin/bash
#SBATCH -p qcb
#SBATCH -A nmherrer_110
#SBATCH -t 1-00:00:00
#SBATCH --mem=16gb
#SBATCH --cpus-per-task=8
#SBATCH --out=sub_out
#SBATCH --error=sub_err

module load python
python sub_python

"""

with open('batch-submit-test_script.slurm','w') as fp:
    fp.write(batch_submit)


Now that our *process* and *submission* scripts are written in a template format, we need to write a new script that will manage the creation of unique copies of these template files set up to run each of our condition sets separately. We can call this a jobs or runs *spawn* script. It spawns all the files to be run later. Pay closest attention to the *sed* commands towards the end of this file, this is where we are actually making our subsitutions in our template files.

In [ ]:
spawn_jobs="""
#!/bin/bash

here=`pwd`
jobslist=${here}'/conditions.txt' # can change this to title of your joblist file
there='Runs'

#------Create run folder and move unique files into the run folder----------------------------------#
if [[ -d ${there} ]]
then
rm -r ${there}
mkdir ${there}

else
mkdir ${there}

fi

cp ${jobslist} ${there}

#-------------- Determine number of jobs you need to run  ----------------
#---------------------------------#
num_jobs=`wc -l ${jobslist} | awk '{print $1 }'`

echo 'Number of runs: ' ${num_jobs} '...'

#-------------- Loop over all polys (start after headed line)  ----------------------------#
iter=1

while [ ${iter} -lt ${num_jobs} ]
do
   let iter=${iter}+1
   curr_var=`head -${iter} ${jobslist} | tail -1 | cut -f 1`
   runname="run-${curr_var}-code"

   #This is not recommended but for this example I am explicitly extracting all
   #of the variables from the jobslist file
   person=`head -${iter} ${jobslist} | tail -1 | cut -f 1`
   commute=`head -${iter} ${jobslist} | tail -1 | cut -f 2`
   
   ####
   
   echo 'Run #'${iter}': creating files for '${runname}'...'
   
   #Create destination file names
   genPY=${there}"/run-${curr_var}-code"'.py'
   genSLURM=${there}'/submit-'${curr_var}'.slurm'
   
   #---- copy template and data files to runs folder 
cp -f ${here}'/batch-code.py' ${genPY}
cp -f ${here}'/batch-submit-test_script.slurm' ${genSLURM}
   # # # NOTE: This is currently making all files in the /Runs/ directory.
   # # # You can split the files to have them save in separate folders per job as well.
   
# ------- Editing the python code file ------- #
   joboutname=${runname}'_out'
   joberrname=${runname}'_err'
   
   sed -i -e s@sub_person@${person}@g\
          -e s@sub_commute@${commute}@g    ${genPY}
  
            
# ------- Editing the Slurm files ------- #
joboutname=${there}'/'${runname}'_out'
joberrname=${there}'/'${runname}'_err'

sed -i -e s@sub_out@${joboutname}@g  \
       -e s@sub_err@${joberrname}@g  \
       -e s@sub_python@${genPY}@g    ${genSLURM}

echo 'Run files successfully modified'
  
done

"""

with open('spawn-runs.sh','w') as fp:
    fp.write(spawn_jobs)


With our *spawn* script in hand, we finally need to write a short script that will read all of the new SLURM files we have generated, and individually submit them to CARC to be run.

In [ ]:
job_submitter="""
#!/bin/sh

# # # THIS SUBMITTER FILE WILL READ FROM YOUR joblist.txt FILE. # # #

here=`pwd`
joblist=${here}'/conditions.txt'
there=${here}'/Runs'


#-------------- Determine the number of jobs to run based on file list  --------------------
#-----------------------------#
num_jobs=`wc -l ${joblist} | awk '{print $1 }'`

echo 'Number of runs to submit: ' ${num_jobs} '...'


#-------------- Loop over all polys (start after headed line)  ----------------------------#
iter=0

while [ ${iter} -lt ${num_jobs} ]
do
   let iter=${iter}+1
   curr_var=`head -${iter} ${joblist} | tail -1 | cut -f 1`
   runname="run-${curr_var}-code"
   genSLURM=${there}'/submit-'${curr_var}'.slurm' ### Be sure this location for your .slurm files is correct. ###

echo "Running ${runname} under job name ${runname}"

sbatch ${genSLURM} ### This is the command that actually submits your jobs. ###

runconfirm="Run # ${iter} : Job name ${runname} - submitted..."
  
echo ${runconfirm}
       
done

"""

with open('jobs-submitter.sh','w') as fp:
    fp.write(job_submitter)


Now we can first call the job/run spawner script, which will use our conditions file to generate unique copies of our process and submission script modified accordingly in a newly created *Runs* folder. Setting a *Runs* folder helps with avoiding creating huge file clutter in the directory with your master template files. Then, we can call our submission manager script, which will separately run each of our SLURM files using an *sbatch* command call.

In [ ]:
%% bash
./spawn-runs.sh
./jobs-submitter.sh